# 06 · Routing analysis and the adaptive-vs-uniform ablation

HeteroSC's central claim is that *how* the experts are combined matters, not only that they are combined. This notebook collects the
diagnostics for all three tasks from the outputs of notebooks `03–05`:

1. **Mean routing weight per expert** (does the dominant expert change across datasets / tasks?),
2. **Normalised routing entropy** (Eq. 9): 1 = even usage, 0 = one expert takes everything,
3. **Adaptive vs uniform** performance under otherwise identical settings.

Read routing weights as *predictive contributions under the training objective*. They are not intrinsic measures of expert quality and not
evidence of biological specialisation.

In [ ]:
from pathlib import Path
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from heterosc import normalized_entropy
from heterosc.plots import plot_routing_by_group

EXPERTS = ["geneformer", "scgpt", "cellfm", "scfoundation"]

## Annotation: routing per dataset (from `test_routing_alpha.npy`)

In [ ]:
rows = []
for ds in ["Pancreas", "Skin", "Liver", "Lung", "Myeloid"]:
    f = Path("outputs/annotation") / ds / "heterosc_adaptive" / "test_routing_alpha.npy"
    if f.exists():
        a = np.load(f)
        rows.append({"dataset": ds, **dict(zip(EXPERTS, a.mean(0).round(3))), "top expert": EXPERTS[int(a.mean(0).argmax())],
                     "normalised entropy": round(normalized_entropy(a.mean(0)), 3)})
pd.DataFrame(rows).set_index("dataset") if rows else "Run notebook 03 for each dataset first."

## Annotation: adaptive vs uniform (from `test_metrics.json`)

In [ ]:
rows = []
for ds in ["Pancreas", "Skin", "Liver", "Lung", "Myeloid"]:
    r = {"dataset": ds}
    for tag in ("heterosc_adaptive", "heterosc_uniform"):
        f = Path("outputs/annotation") / ds / tag / "test_metrics.json"
        if f.exists():
            r[tag] = json.loads(f.read_text())["macro_f1"]
    rows.append(r)
df = pd.DataFrame(rows).set_index("dataset")
if "heterosc_adaptive" in df and "heterosc_uniform" in df:
    df["gain"] = df["heterosc_adaptive"] - df["heterosc_uniform"]
df.round(4)

## Drug response: routing weights from a trained fold

In [ ]:
import torch
from heterosc.data import load_drug_tables, load_cell_line_embeddings, keep_complete_cell_lines
from heterosc.drug_response import DrugFeatureStore, DrugResponseConfig, DrugResponseModel, collect_routing

FOLD, RUN = 0, Path("outputs/drug_response/heterosc_gcn")
if (RUN / f"fold{FOLD}_model.pt").exists():
    resp, graphs, fp = load_drug_tables("data/drug_response")
    emb = load_cell_line_embeddings("outputs/drug_response_embeddings", EXPERTS)
    resp = keep_complete_cell_lines(resp, emb)
    dims = {e: len(next(iter(emb[e].values()))) for e in EXPERTS}
    cfg = DrugResponseConfig(cell_encoder="heterosc", drug_mode="gcn", expert_dims=dims, device="cpu")
    store = DrugFeatureStore(graphs, fp)
    model = DrugResponseModel("heterosc", "gcn", dims, store.fp_dim, cfg.d_shared)
    model.load_state_dict(torch.load(RUN / f"fold{FOLD}_model.pt", map_location="cpu"))
    a = collect_routing(model, resp[resp.fold == FOLD], emb, store, cfg)
    print(a[EXPERTS].mean().round(3).to_dict(), "| normalised entropy", round(normalized_entropy(a[EXPERTS].mean().values), 3))
    a[EXPERTS].plot(kind="box", figsize=(6, 3.2), title="Routing weight across test pairs (fold %d)" % FOLD); plt.show()
else:
    print("Train a fold in notebook 04 first.")

## Reporting checklist
* Use the **same data partitions, seeds and training settings** for adaptive and uniform runs.
* Report weights **and** entropy, and say which regulariser was active (diversity for annotation, per-input entropy for drug response, none for perturbation).
* Never describe a routing weight as the "importance" or "quality" of a foundation model.